**【プログラムの仕様】**  

＜インプット＞  
・月別売上.xlsx  
・社員リスト.xlsx

＜アウトプット＞  
・総売上内訳_田中.xlsx  
・総売上内訳_佐藤.xlsx  
・総売上内訳_鈴木.xlsx  
・総売上内訳_中村.xlsx  
（各ファイルの中身は、月別売上.xlsxの全シートの売上金額を担当者ごとに集計し、行：取引先、列：商品名として表にまとめたもの）

【パーツごとに分解して考える】  
①. ４月・５月・６月各シートのデータを結合する  
②. 売上金額の列を追加する  
③. 担当者名の列を追加する  
④. 担当者別にデータを分ける  
⑤. 行：取引先、列：商品名としてピボットテーブルを作成して出力  

In [ ]:
# ①. ４月・５月・６月各シートのデータを結合する

import pandas as pd

DF=pd.ExcelFile("月別売上.xlsx")
month_list=DF.sheet_names
# month_list=["4月", "5月", "6月"]

df_concat=pd.DataFrame()

for month in month_list:
    df_month=pd.read_excel("月別売上.xlsx", sheet_name=month)
    df_concat=pd.concat([df_concat, df_month])
    
df_concat

In [ ]:
DF=pd.ExcelFile("月別売上.xlsx")
DF.sheet_names

In [ ]:
# ②. 売上金額の列を追加する

df_concat["売上金額"]=df_concat["単価"]*df_concat["個数"]
df_concat

In [ ]:
# ③. 担当者名の列を追加する

df_syain=pd.read_excel("社員リスト.xlsx")
df_syain

In [ ]:
df_syain["社員番号"]=df_syain["社員番号"].str.replace("A-", "").str.replace("B-", "")
# df_syain["社員番号"]=df_syain["社員番号"].str.replace("[A-Z]-", "", regex=True)
df_syain

In [ ]:
df_merge=pd.merge(df_concat, df_syain, on="社員番号")
df_merge

In [ ]:
type(df_concat.iloc[0,3])

In [ ]:
type(df_syain.iloc[0,0])

In [ ]:
df_syain=df_syain.astype({"社員番号":int})
df_syain

In [ ]:
type(df_syain.iloc[0,0])

In [ ]:
df_merge=pd.merge(df_concat, df_syain, on="社員番号")
df_merge

In [ ]:
# ④. 担当者別にデータを分ける

df_each=df_merge[df_merge["名前"]=="田中"]
df_each

In [ ]:
# ⑤. 行：取引先、列：商品名としてピボットテーブルを作成して出力

df_output=df_each.pivot_table(index="取引先", columns="商品名", values="売上金額", aggfunc="sum", fill_value=0, margins=True, margins_name="総計")
df_output

In [ ]:
# 完成版

import pandas as pd

DF=pd.ExcelFile("月別売上.xlsx")
month_list=DF.sheet_names

df_concat=pd.DataFrame()

for month in month_list:
    df_month=pd.read_excel("月別売上.xlsx", sheet_name=month)
    df_concat=pd.concat([df_concat, df_month])
    
df_concat["売上金額"]=df_concat["単価"]*df_concat["個数"]

df_syain=pd.read_excel("社員リスト.xlsx", sheet_name="リスト")
df_syain["社員番号"]=df_syain["社員番号"].str.replace("A-", "").str.replace("B-", "")
df_syain=df_syain.astype({"社員番号":int})

df_merge=pd.merge(df_concat, df_syain, on="社員番号")

name_list=df_merge["名前"].unique()

for name in name_list:
    df_each=df_merge[df_merge["名前"]==name]
    df_output=df_each.pivot_table(index="取引先", columns="商品名", values="売上金額", aggfunc="sum", fill_value=0, margins=True, margins_name="総計")
    df_output.to_excel("総売上内訳_{}.xlsx".format(name))